# IHES Method 10 — Immediate-inverse pruning

Remove only the exact inverse of the previous move.

This notebook is one of thirteen controlled experiments. It runs a small smoke search first and then the requested T4 benchmark for puzzle IDs 100–120 at beam width 365000. Every accepted path is replayed with the 18 official generators. The final `submission.csv` is replay-validated for all 1,003 competition rows.

The notebook never contains credentials, competition data, participant submissions, or model weights. All substantial external ideas and licenses are documented in the public repository.


In [ ]:
from pathlib import Path

METHOD_ID = 10
RUN_SMOKE = True
RUN_FULL = True
SMOKE_PUZZLE_IDS = (100,)
FULL_PUZZLE_IDS = tuple(range(100, 121))
SMOKE_BEAM_WIDTH = 64
FULL_BEAM_WIDTH = 365_000
DEVICE = "cuda"
ASSET_ROOT = Path("/kaggle/input")
MODEL_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
REFERENCE_SUBMISSION = None  # Optional attached replay-valid control CSV.


In [ ]:
import shutil
import subprocess
import sys

repository_url = 'https://github.com/cheldieva-l/ihes-13-methods'
checkout = WORKING / "ihes-13-methods"
if checkout.exists():
    shutil.rmtree(checkout)
subprocess.run(
    ["git", "clone", "--depth", "1", repository_url, str(checkout)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(checkout), "--no-deps", "-q"],
    check=True,
)
if METHOD_ID == 12:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "kociemba>=1.2", "-q"],
        check=True,
    )
sys.path.insert(0, str(checkout))
repository_commit = subprocess.check_output(
    ["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True
).strip()
print({"repository_commit": repository_commit, "method_id": METHOD_ID})


In [ ]:
import json
import torch
from ihes13.runner import load_method_config, run_experiment

method_config = load_method_config(METHOD_ID)
print(json.dumps(method_config, indent=2, sort_keys=True))
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("A Kaggle GPU session is required for the requested benchmark")
print({
    "torch": torch.__version__,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "asset_root": str(ASSET_ROOT),
})


In [ ]:
smoke_summary = None
if RUN_SMOKE:
    smoke_summary = run_experiment(
        METHOD_ID,
        asset_root=ASSET_ROOT,
        model_root=MODEL_ROOT,
        output_root=WORKING / f"method-{METHOD_ID:02d}" / "smoke",
        puzzle_ids=SMOKE_PUZZLE_IDS,
        beam_width=SMOKE_BEAM_WIDTH,
        device=DEVICE,
        smoke=True,
        reference_submission=REFERENCE_SUBMISSION,
    )
    print("Smoke summary")
    print(json.dumps(smoke_summary, indent=2, sort_keys=True))


In [ ]:
full_summary = None
if RUN_FULL:
    full_output = WORKING / f"method-{METHOD_ID:02d}" / "full"
    full_summary = run_experiment(
        METHOD_ID,
        asset_root=ASSET_ROOT,
        model_root=MODEL_ROOT,
        output_root=full_output,
        puzzle_ids=FULL_PUZZLE_IDS,
        beam_width=FULL_BEAM_WIDTH,
        device=DEVICE,
        smoke=False,
        reference_submission=REFERENCE_SUBMISSION,
    )
    shutil.copy2(full_output / "submission.csv", WORKING / "submission.csv")
    shutil.copy2(full_output / "benchmark_rows.json", WORKING / f"method_{METHOD_ID:02d}_benchmark_rows.json")
    shutil.copy2(full_output / "summary.json", WORKING / f"method_{METHOD_ID:02d}_summary.json")
    print("Full benchmark summary")
    print(json.dumps(full_summary, indent=2, sort_keys=True))


In [ ]:
from ihes_dual.assets import find_competition_assets
from ihes_dual.puzzle import IHESPuzzle
from ihes_dual.submission import validate_submission

final_submission = WORKING / "submission.csv"
if not final_submission.is_file():
    source = WORKING / f"method-{METHOD_ID:02d}" / "smoke" / "submission.csv"
    shutil.copy2(source, final_submission)
assets = find_competition_assets(ASSET_ROOT)
puzzle = IHESPuzzle.from_puzzle_info(assets.puzzle_info)
validation = validate_submission(final_submission, assets.test_csv, puzzle)
print({"submission": str(final_submission), "validation": validation})
if full_summary is not None and not full_summary.get("completed", False):
    raise RuntimeError(
        "The full benchmark recorded failed puzzle runs; inspect "
        f"method_{METHOD_ID:02d}_benchmark_rows.json"
    )
